In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Constants
MAX_FEATURES = 10000  # Reduced vocabulary size
MAX_LEN = 200
EMBEDDING_DIM = 64  # Reduced embedding dimensions
LSTM_UNITS = 32  # Reduced LSTM units
DENSE_UNITS = 16  # Reduced dense layer units
DROPOUT_RATE = 0.4

print("Loading data...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=MAX_FEATURES)
x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)

print("Creating model...")
model = Sequential([
    Embedding(MAX_FEATURES, EMBEDDING_DIM, input_length=MAX_LEN),
    LSTM(LSTM_UNITS),
    Dense(DENSE_UNITS, activation='relu'),
    Dropout(DROPOUT_RATE),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Training model...")
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

print("Evaluating model...")
_, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}")

print("Saving model...")
model.save('sentiment_model.h5', include_optimizer=False)  # Exclude optimizer to reduce size

print("Saving tokenizer...")
word_index = imdb.get_word_index()
tokenizer = {k: (v + 3) for k, v in word_index.items() if v < MAX_FEATURES}  # Limit vocabulary
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Model and tokenizer saved successfully.")


Loading data...
Creating model...
Training model...
Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.6682 - loss: 0.5933 - val_accuracy: 0.8552 - val_loss: 0.3363
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.8863 - loss: 0.3054 - val_accuracy: 0.8596 - val_loss: 0.3248
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9240 - loss: 0.2202 - val_accuracy: 0.8578 - val_loss: 0.3518
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9263 - loss: 0.2109 - val_accuracy: 0.8640 - val_loss: 0.3559
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9395 - loss: 0.1753 - val_accuracy: 0.8650 - val_loss: 0.3992
Evaluating model...


Test Accuracy: 0.8625
Saving model...
Saving tokenizer...
Model and tokenizer saved successfully.
